# Module 16: XVA & Counterparty Credit Risk

This notebook verifies the implementation of the XVA mathematical models.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from xva.exposure import simulate_exposures
from xva.xva import calculate_cva, calculate_dva, calculate_fva

np.random.seed(42)

## 1. Exposure Simulation

Simulate Mark-to-Market (MtM) paths and calculate EE, PFE (95%), and ENE.

In [ ]:
num_paths = 1000
num_steps = 10

# Simulate Brownian motion for MtM
dt = 1.0
dW = np.random.normal(0, np.sqrt(dt), (num_paths, num_steps))
mtm_paths = np.cumsum(dW, axis=1) * 10 + 5.0 # drift and scale

ee, pfe, ene = simulate_exposures(mtm_paths)

time_grid = np.arange(1, num_steps + 1)

plt.figure(figsize=(10, 6))
plt.plot(time_grid, ee, label='Expected Exposure (EE)', color='blue')
plt.plot(time_grid, pfe, label='Potential Future Exposure (PFE 95%)', color='red', linestyle='--')
plt.plot(time_grid, ene, label='Expected Negative Exposure (ENE)', color='green')
plt.title('Simulated Exposure Profiles')
plt.xlabel('Time Step')
plt.ylabel('Exposure')
plt.legend()
plt.grid(True)
plt.show()

## 2. Value Adjustments (XVA)

Calculate CVA, DVA, and FVA using the simulated exposures.

In [ ]:
# Assumptions
pd = np.full(num_steps, 0.02) # Marginal PD of counterparty
pd_own = np.full(num_steps, 0.015) # Marginal PD of own firm
lgd = 0.6
lgd_own = 0.6
df = np.exp(-0.02 * time_grid) # Discount factors
fca_spread = 0.015
fba_spread = 0.010

cva = calculate_cva(ee, pd, lgd, df)
dva = calculate_dva(ene, pd_own, lgd_own, df)
fva = calculate_fva(ee, ene, fca_spread, fba_spread, df, dt=dt)

print(f"Credit Valuation Adjustment (CVA): {cva:.2f}")
print(f"Debt Valuation Adjustment (DVA): {dva:.2f}")
print(f"Funding Valuation Adjustment (FVA): {fva:.2f}")